# Deterministic Text Generation for Election Night Pre-Write

In [1]:
import requests
from datetime import datetime
import pandas as pd
import json

In [2]:
data = pd.read_csv("../data/proposal_4_results_2025.csv")

In [3]:
data.head()

,district_combo,name,proposal_number,YES,NO,yes_pct,no_pct,dw_category
0,52075,Prospect Heights,Proposal Number 4,394,216,64.6,35.4,61-80%
1,44008,Prospect Heights,Proposal Number 4,443,243,64.6,35.4,61-80%
2,44009,Prospect Heights,Proposal Number 4,434,208,67.6,32.4,61-80%
3,44060,Prospect Heights,Proposal Number 4,140,86,61.9,38.1,61-80%
4,44005,Prospect Heights,Proposal Number 4,174,67,72.2,27.8,61-80%


In [4]:
data.dtypes

district_combo       int64
name                object
proposal_number     object
YES                  int64
NO                   int64
yes_pct            float64
no_pct             float64
dw_category         object
dtype: object

In [5]:
data['district_combo'] = data['district_combo'].astype(str)

In [6]:
data.dtypes

district_combo      object
name                object
proposal_number     object
YES                  int64
NO                   int64
yes_pct            float64
no_pct             float64
dw_category         object
dtype: object

In [7]:
# creating a column for assembly district from the district_combo
data["assembly_district"] = (
    data["district_combo"]
    .str.extract(r"^(\d{2})", expand=False)
    .astype("Int64")
)

In [8]:
# creates a range for all of the borough and assembly districts
borough_ranges = { 
    "Manhattan": range(65, 76),
    "Bronx": range(77, 88),
    "Brooklyn": range(41, 61),
    "Queens": range(23, 41),
    "Staten Island": range(61, 65)
}

In [9]:
assembly_to_borough = { 
    district: borough
    for borough, districts in borough_ranges.items()
    for district in districts

}

In [10]:
data["borough"] = (
    data["assembly_district"]
    .map(assembly_to_borough)
    .fillna("Unknown")
)

In [11]:
data.head()

,district_combo,name,proposal_number,YES,NO,yes_pct,no_pct,dw_category,assembly_district,borough
0,52075,Prospect Heights,Proposal Number 4,394,216,64.6,35.4,61-80%,52,Brooklyn
1,44008,Prospect Heights,Proposal Number 4,443,243,64.6,35.4,61-80%,44,Brooklyn
2,44009,Prospect Heights,Proposal Number 4,434,208,67.6,32.4,61-80%,44,Brooklyn
3,44060,Prospect Heights,Proposal Number 4,140,86,61.9,38.1,61-80%,44,Brooklyn
4,44005,Prospect Heights,Proposal Number 4,174,67,72.2,27.8,61-80%,44,Brooklyn


In [12]:
totals = (
    data.groupby(["name", "borough"])[["YES", "NO"]]
    .sum()
    .reset_index()
)

neighborhood_data = totals.iloc[0].to_dict()

In [13]:
neighborhood_data

{'name': 'Annadale', 'borough': 'Staten Island', 'YES': 4106, 'NO': 13342}

In [14]:
totals["pct_yes"] = totals["YES"] / (totals["YES"] + totals["NO"])


In [15]:
neighborhood_data = json.loads(totals.reset_index().iloc[0].to_json())

In [16]:
neighborhood_data

{'index': 0,
 'name': 'Annadale',
 'borough': 'Staten Island',
 'YES': 4106,
 'NO': 13342,
 'pct_yes': 0.2353278313}

In [17]:
# if neighborhood_data.get("pct_yes") >= 0.75:
#     text = f'{neighborhood_data.get("name")} voted overwhelmingly to approve Proposal Number 4, with {neighborhood_data.get("pct_yes")}% of the vote.'
# else:
#     text = f'{neighborhood_data.get("name")} voted to reject Proposal Number 4, with {neighborhood_data.get("pct_yes")*100:.2f}% of the vote.'

In [18]:
# if neighborhood_data.get("pct_yes") >= 0.50:
#     text = f'{neighborhood_data.get("name")} voted to approve Proposal Number 4, with {neighborhood_data.get("pct_yes")}% of the vote.'
# else:
#     text = f'{neighborhood_data.get("name")} voted to reject Proposal Number 4, with {neighborhood_data.get("pct_yes")*100:.2f}% of the vote.'if neighborhood_data.get(

In [19]:
# if neighborhood_data.get("pct_yes") >= 0.25:
#     text = f'{neighborhood_data.get("name")} voted overwhelmingly to approve Proposal Number 4, with {neighborhood_data.get("pct_yes")}% of the vote.'
# else:
#     text = f'{neighborhood_data.get("name")} voted to approve Proposal Number 4, with {neighborhood_data.get("pct_yes")*100:.2f}% of the vote.'

### Sentence 1

In [20]:
pct_yes = neighborhood_data.get("pct_yes")
name = neighborhood_data.get("name")
pct_yes_fmt = pct_yes * 100  # Converts to percentage

if pct_yes >= 0.75:
    sentence1 = (
        f"{name} voted overwhelmingly to approve Proposal Number 4, "
        f"with {pct_yes_fmt:.2f}% of the vote."
    )
elif pct_yes >= 0.50:
    sentence1 = (
        f"{name} voted to approve Proposal Number 4, "
        f"with {pct_yes_fmt:.2f}% of the vote."
    )
elif pct_yes >= 0.25:
    sentence1 = (
        f"{name} narrowly voted to reject Proposal Number 4, "
        f"with {pct_yes_fmt:.2f}% voting yes."
    )
else:
    sentence1 = (
        f"{name} voted overwhelmingly to reject Proposal Number 4, "
        f"with only {pct_yes_fmt:.2f}% voting yes."
    )

print(sentence1)


Annadale voted overwhelmingly to reject Proposal Number 4, with only 23.53% voting yes.


### Sentence 2

In [21]:
yes_votes = neighborhood_data.get("YES")
no_votes = neighborhood_data.get("NO")
total_votes = yes_votes + no_votes

sentence2 = f"The margin of votes was {abs(yes_votes - no_votes)} out of {total_votes} total ballots cast."

In [22]:
sentence2

'The margin of votes was 9236 out of 17448 total ballots cast.'

### Sentence 3

In [23]:
if pct_yes >= 0.75:
    sentence3 = "Support for the ballot measure that would create an affordable housing appeals board consisting of the mayor, Council speaker, and local borough president, signaling strong approval among voters."
elif pct_yes <= 0.25:
    sentence3 = "Opposition was overwhelming, indicating strong resistance to the measure to create an affordable housing appeals board consisting of the mayor, Council speaker, and local borough president."
else:
    sentence3 = "The result shows moderate support for the measure to createan affordable housing appeals board consisting of the mayor, Council speaker, and local borough president, reflecting a divided electorate."


In [24]:
sentence3

'Opposition was overwhelming, indicating strong resistance to the measure to create an affordable housing appeals board consisting of the mayor, Council speaker, and local borough president.'

### Sentence 4

In [25]:
# the sum of the borough votes
borough_totals = data.groupby("borough")[["YES", "NO"]].sum()
borough_totals

,YES,NO
borough,,
Bronx,882367,365520
Brooklyn,2156084,1554683
Manhattan,1698109,1090611
Queens,1663248,1209267
Staten Island,348100,580750
Unknown,191253,140274


In [26]:
# finding the borough average and turning it into a dictionary
borough_totals["borough_pct_yes"] = borough_totals["YES"] / (borough_totals["YES"] + borough_totals["NO"])

borough_avg_yes = borough_totals["borough_pct_yes"].to_dict()
borough_avg_yes

{'Bronx': 0.707088863014039,
 'Brooklyn': 0.5810345947347273,
 'Manhattan': 0.6089205800510629,
 'Queens': 0.5790215194698722,
 'Staten Island': 0.37476449372880444,
 'Unknown': 0.5768851405767856}

In [34]:
borough = neighborhood_data.get("borough", "the area")
borough_pct_yes = borough_avg_yes.get(borough, None)
borough_pct_yes_fmt = borough_pct_yes * 100 if borough_pct_yes is not None else None

sentence4 = "" 
if borough_pct_yes is not None:
    if pct_yes > borough_pct_yes:
        sentence4e = (
            f"This neighborhood’s support for Proposal 4 at {pct_yes_fmt:.1f}% was "
            f"higher than the {borough} borough average of {borough_pct_yes_fmt:.1f}%."
        )
    elif pct_yes < borough_pct_yes:
        sentence4 = (
            f"This neighborhood’s support at {pct_yes_fmt:.1f}% was "
            f"lower than the {borough} borough average of {borough_pct_yes_fmt:.1f}%."
        )
    else:
        sentence4 = (
            f"The neighborhood’s support matched the {borough} borough average of {borough_pct_yes_fmt:.1f}%."
        )


In [36]:
sentence4

'This neighborhood’s support at 23.5% was lower than the Staten Island borough average of 37.5%.'

In [37]:
paragraph1 = " ".join([sentence1, sentence2, sentence3, sentence4])

In [38]:
paragraph1

'Annadale voted overwhelmingly to reject Proposal Number 4, with only 23.53% voting yes. The margin of votes was 9236 out of 17448 total ballots cast. Opposition was overwhelming, indicating strong resistance to the measure to create an affordable housing appeals board consisting of the mayor, Council speaker, and local borough president. This neighborhood’s support at 23.5% was lower than the Staten Island borough average of 37.5%.'